# ML Sentiment and Emotional Analysis in English Literature

## IOMP Batch A16 - Enhanced Hybrid Deep Learning Architecture

### Architecture:
- **BERT-base** (110M params) - Contextual embeddings
- **CNN** - Local feature extraction (multi-scale: 3,4,5 kernels)
- **BiLSTM** - Long-range dependency modeling
- **Self-Attention** - Focus on emotionally significant tokens

### Emotion Categories:
- **Joy** | **Sadness** | **Anger** | **Fear** | **Surprise** | **Neutral**

### Sentiment Polarity:
- **Positive** | **Negative** | **Neutral**

### Datasets:
- **Abdallah Wagih Emotion Dataset** (Kaggle)
- **ISEAR Dataset** (Kaggle)

---

## 1. Check GPU & Setup

In [ ]:
!nvidia-smi

import torch
print(f"\nPyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
    print("\nGPU ready!")
else:
    print("\nNO GPU! Go to Runtime -> Change runtime type -> GPU")
    raise RuntimeError("GPU required")

## 2. Install Dependencies

In [ ]:
%%capture
!pip install -q transformers torch torchvision
!pip install -q nltk scikit-learn matplotlib seaborn tqdm pyyaml
!pip install -q kagglehub

import nltk
for pkg in ['punkt', 'stopwords', 'wordnet', 'averaged_perceptron_tagger']:
    nltk.download(pkg, quiet=True)

print("All dependencies installed!")

## 3. Model Architecture

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import BertModel

class SelfAttention(nn.Module):
    def __init__(self, hidden_size, dropout=0.1):
        super().__init__()
        self.W = nn.Linear(hidden_size, hidden_size)
        self.v = nn.Linear(hidden_size, 1, bias=False)
        self.dropout = nn.Dropout(dropout)
    
    def forward(self, lstm_output, mask=None):
        # lstm_output: [batch, seq_len, hidden]
        energy = torch.tanh(self.W(lstm_output))
        scores = self.v(energy).squeeze(-1)  # [batch, seq_len]
        
        if mask is not None:
            # Convert mask to float32 and use -1e4 (safe for FP16)
            scores = scores.float()
            mask = mask.float()
            scores = scores.masked_fill(mask == 0, -1e4)
        
        attention_weights = F.softmax(scores, dim=1)
        attention_weights = self.dropout(attention_weights)
        context = torch.sum(attention_weights.unsqueeze(-1) * lstm_output.float(), dim=1)
        return context, attention_weights


class EnhancedSentimentModel(nn.Module):
    def __init__(self):
        super().__init__()
        
        print("Loading BERT-base-uncased...")
        self.bert = BertModel.from_pretrained('bert-base-uncased')
        bert_size = 768
        
        # Freeze first 2 layers
        for i in range(2):
            for param in self.bert.encoder.layer[i].parameters():
                param.requires_grad = False
        
        # CNN
        self.convs = nn.ModuleList([
            nn.Conv1d(bert_size, 256, 3),
            nn.Conv1d(bert_size, 512, 4),
            nn.Conv1d(bert_size, 1024, 5)
        ])
        cnn_out = 256 + 512 + 1024
        
        # BiLSTM
        self.lstm = nn.LSTM(
            bert_size, 512, 
            num_layers=3, 
            bidirectional=True, 
            dropout=0.2, 
            batch_first=True
        )
        lstm_out = 512 * 2
        
        self.attention = SelfAttention(lstm_out, 0.1)
        combined = cnn_out + lstm_out
        
        # Classifiers
        self.sentiment_clf = nn.Sequential(
            nn.Linear(combined, 512),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(512, 3)
        )
        
        self.emotion_clf = nn.Sequential(
            nn.Linear(combined, 512),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(512, 6)
        )
        
        self.dropout = nn.Dropout(0.2)
    
    def forward(self, input_ids, attention_mask):
        bert_out = self.bert(input_ids, attention_mask).last_hidden_state
        
        # CNN path (convert to float32 for stability)
        x = bert_out.float().transpose(1, 2)
        cnn_feats = []
        for conv in self.convs:
            h = torch.relu(conv(x))
            h = torch.max_pool1d(h, h.size(2)).squeeze(2)
            cnn_feats.append(h)
        cnn_out = torch.cat(cnn_feats, 1)
        cnn_out = self.dropout(cnn_out)
        
        # LSTM + Attention path
        lstm_out, _ = self.lstm(bert_out.float())
        lstm_out = self.dropout(lstm_out)
        context, attn_weights = self.attention(lstm_out, attention_mask)
        
        combined = torch.cat([cnn_out, context], 1)
        
        sentiment = self.sentiment_clf(combined)
        emotion = self.emotion_clf(combined)
        
        return sentiment, emotion, attn_weights

print("Model classes defined!")

## 4. Download & Prepare Datasets

In [ ]:
import kagglehub
import pandas as pd
import numpy as np
import os
from sklearn.model_selection import train_test_split

print("Downloading Kaggle Datasets...")
print("="*60)

# ===================== WAGIH DATASET =====================
print("\n[1/2] Downloading Wagih Emotion Dataset...")
path_wagih = kagglehub.dataset_download("abdallahwagih/emotion-dataset")
wagih_files = os.listdir(path_wagih)
print(f"    Files: {wagih_files}")

wagih_csv = [f for f in wagih_files if f.endswith('.csv')][0]
df_wagih = pd.read_csv(os.path.join(path_wagih, wagih_csv))
print(f"    Samples: {len(df_wagih)}")
print(f"    Columns: {df_wagih.columns.tolist()}")

# Detect columns
w_emo = 'Emotion' if 'Emotion' in df_wagih.columns else df_wagih.columns[1]
w_txt = 'Comment' if 'Comment' in df_wagih.columns else df_wagih.columns[0]
print(f"    Emotions: {df_wagih[w_emo].unique()}")

def map_wagih(row):
    vec = np.zeros(6, dtype=np.float32)
    sent = 1
    lbl = str(row[w_emo]).strip().lower()
    
    if lbl in ['joy', 'love', 'happy']:
        vec[0] = 1; sent = 2
    elif lbl in ['sadness', 'sad']:
        vec[1] = 1; sent = 0
    elif lbl in ['anger', 'angry']:
        vec[2] = 1; sent = 0
    elif lbl in ['fear', 'afraid']:
        vec[3] = 1; sent = 0
    elif lbl in ['surprise']:
        vec[4] = 1; sent = 2
    else:
        vec[5] = 1; sent = 1
    
    return pd.Series([str(row[w_txt]), vec, int(sent)])

print("    Processing...")
df_w = df_wagih.apply(map_wagih, axis=1)
df_w.columns = ['text', 'emotions', 'sentiment']
print(f"    Processed: {len(df_w)} samples")

# ===================== ISEAR DATASET =====================
print("\n[2/2] Downloading ISEAR Dataset...")
path_isear = kagglehub.dataset_download("faisalsanto007/isear-dataset")
isear_files = os.listdir(path_isear)
print(f"    Files: {isear_files}")

# Find CSV
isear_csv = None
for f in isear_files:
    fp = os.path.join(path_isear, f)
    if f.endswith('.csv'):
        isear_csv = fp
    elif os.path.isdir(fp):
        for sf in os.listdir(fp):
            if sf.endswith('.csv'):
                isear_csv = os.path.join(fp, sf)

if isear_csv:
    print(f"    Loading: {isear_csv}")
    df_isear = pd.read_csv(isear_csv, on_bad_lines='skip')
    print(f"    Columns: {df_isear.columns.tolist()}")
    print(f"    Samples: {len(df_isear)}")
    
    # ISEAR: 'sentiment' = emotion label, 'content' = text
    emo_col = 'sentiment'
    txt_col = 'content'
    print(f"    Unique emotions: {df_isear[emo_col].unique()}")
    
    def map_isear(row):
        vec = np.zeros(6, dtype=np.float32)
        sent = 1
        lbl = str(row[emo_col]).strip().lower()
        txt = str(row[txt_col]) if pd.notna(row[txt_col]) else ""
        
        if len(txt) < 10:
            return None
        
        if 'joy' in lbl or 'happy' in lbl:
            vec[0] = 1; sent = 2
        elif 'sad' in lbl:
            vec[1] = 1; sent = 0
        elif 'ang' in lbl:
            vec[2] = 1; sent = 0
        elif 'fear' in lbl:
            vec[3] = 1; sent = 0
        elif 'shame' in lbl or 'guilt' in lbl or 'disgust' in lbl:
            vec[1] = 1; sent = 0
        else:
            return None
        
        return pd.Series([txt, vec, int(sent)])
    
    print("    Processing...")
    df_i = df_isear.apply(map_isear, axis=1).dropna()
    df_i.columns = ['text', 'emotions', 'sentiment']
    print(f"    Processed: {len(df_i)} samples")
else:
    print("    ERROR: No CSV found")
    df_i = pd.DataFrame(columns=['text', 'emotions', 'sentiment'])

# ===================== MERGE & SPLIT =====================
print("\nMerging...")
final_df = pd.concat([df_w, df_i], ignore_index=True)
final_df = final_df.sample(frac=1, random_state=42).reset_index(drop=True)
final_df = final_df.dropna()
final_df = final_df[final_df['text'].str.len() >= 10]

# Ensure sentiment is integer
final_df['sentiment'] = final_df['sentiment'].astype(int)

print(f"Combined: {len(final_df)} samples")

if len(final_df) == 0:
    raise ValueError("No samples!")

train_df, temp_df = train_test_split(final_df, test_size=0.30, random_state=42, stratify=final_df['sentiment'])
val_df, test_df = train_test_split(temp_df, test_size=0.50, random_state=42, stratify=temp_df['sentiment'])

print("\n" + "="*60)
print("DATASET READY!")
print("="*60)
print(f"Train: {len(train_df):,} | Val: {len(val_df):,} | Test: {len(test_df):,}")

print("\nSentiment Distribution:")
for idx, count in train_df['sentiment'].value_counts().sort_index().items():
    print(f"  {['Negative', 'Neutral', 'Positive'][int(idx)]}: {count:,}")

## 5. Create DataLoaders

In [ ]:
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer

class EmotionDataset(Dataset):
    def __init__(self, texts, sentiments, emotions, tokenizer, max_len=256):
        self.texts = list(texts)
        self.sentiments = [int(s) for s in sentiments]
        self.emotions = [np.array(e, dtype=np.float32) for e in emotions]
        self.tokenizer = tokenizer
        self.max_len = max_len
    
    def __len__(self):
        return len(self.texts)
    
    def __getitem__(self, idx):
        enc = self.tokenizer(
            str(self.texts[idx]),
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        return {
            'input_ids': enc['input_ids'].flatten(),
            'attention_mask': enc['attention_mask'].flatten(),
            'sentiment': torch.tensor(self.sentiments[idx], dtype=torch.long),
            'emotions': torch.tensor(self.emotions[idx], dtype=torch.float32)
        }

print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')

MAX_LEN = 256
BATCH_SIZE = 32

train_dataset = EmotionDataset(
    train_df['text'].values,
    train_df['sentiment'].values,
    list(train_df['emotions'].values),
    tokenizer, MAX_LEN
)

val_dataset = EmotionDataset(
    val_df['text'].values,
    val_df['sentiment'].values,
    list(val_df['emotions'].values),
    tokenizer, MAX_LEN
)

test_dataset = EmotionDataset(
    test_df['text'].values,
    test_df['sentiment'].values,
    list(test_df['emotions'].values),
    tokenizer, MAX_LEN
)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

print(f"\nDataLoaders ready!")
print(f"Train batches: {len(train_loader)} | Val: {len(val_loader)} | Test: {len(test_loader)}")

## 6. Initialize Model

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

model = EnhancedSentimentModel().to(device)

total = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"\nTotal params: {total:,} ({total/1e6:.1f}M)")
print(f"Trainable: {trainable:,} ({trainable/1e6:.1f}M)")

In [ ]:
criterion_sentiment = nn.CrossEntropyLoss()
criterion_emotion = nn.BCEWithLogitsLoss()

optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5, weight_decay=0.01)
scaler = torch.amp.GradScaler('cuda')

num_epochs = 10
total_steps = len(train_loader) * num_epochs

scheduler = torch.optim.lr_scheduler.OneCycleLR(
    optimizer, max_lr=2e-5, total_steps=total_steps
)

print(f"Epochs: {num_epochs}")
print(f"Total steps: {total_steps}")
print(f"Mixed Precision: Enabled")

## 7. Training

In [ ]:
from tqdm.auto import tqdm
from sklearn.metrics import accuracy_score

history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}
best_val_loss = float('inf')

print("\n" + "="*70)
print("TRAINING: BERT + CNN + BiLSTM + Attention")
print("="*70)
print(f"Samples: {len(train_df):,} | Batch: {BATCH_SIZE} | Seq: {MAX_LEN}")
print("="*70 + "\n")

for epoch in range(num_epochs):
    print(f"\nEpoch {epoch+1}/{num_epochs}")
    print("-"*40)
    
    # TRAIN
    model.train()
    train_loss = 0
    train_preds, train_labels = [], []
    
    pbar = tqdm(train_loader, desc='Training')
    for batch in pbar:
        input_ids = batch['input_ids'].to(device)
        mask = batch['attention_mask'].to(device)
        sent_labels = batch['sentiment'].to(device)
        emo_labels = batch['emotions'].to(device)
        
        optimizer.zero_grad()
        
        with torch.amp.autocast('cuda'):
            sent_logits, emo_logits, _ = model(input_ids, mask)
            loss = 0.6 * criterion_sentiment(sent_logits, sent_labels) + \
                   0.4 * criterion_emotion(emo_logits, emo_labels)
        
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()
        
        train_loss += loss.item()
        preds = torch.argmax(sent_logits, 1).cpu()
        train_preds.extend(preds.numpy())
        train_labels.extend(sent_labels.cpu().numpy())
        pbar.set_postfix({'loss': f'{loss.item():.4f}'})
    
    avg_train_loss = train_loss / len(train_loader)
    train_acc = accuracy_score(train_labels, train_preds)
    
    # VALIDATE
    model.eval()
    val_loss = 0
    val_preds, val_labels = [], []
    
    with torch.no_grad():
        for batch in tqdm(val_loader, desc='Validating'):
            input_ids = batch['input_ids'].to(device)
            mask = batch['attention_mask'].to(device)
            sent_labels = batch['sentiment'].to(device)
            emo_labels = batch['emotions'].to(device)
            
            with torch.amp.autocast('cuda'):
                sent_logits, emo_logits, _ = model(input_ids, mask)
                loss = 0.6 * criterion_sentiment(sent_logits, sent_labels) + \
                       0.4 * criterion_emotion(emo_logits, emo_labels)
            
            val_loss += loss.item()
            preds = torch.argmax(sent_logits, 1).cpu()
            val_preds.extend(preds.numpy())
            val_labels.extend(sent_labels.cpu().numpy())
    
    avg_val_loss = val_loss / len(val_loader)
    val_acc = accuracy_score(val_labels, val_preds)
    
    history['train_loss'].append(avg_train_loss)
    history['val_loss'].append(avg_val_loss)
    history['train_acc'].append(train_acc)
    history['val_acc'].append(val_acc)
    
    print(f"Train Loss: {avg_train_loss:.4f} | Acc: {train_acc:.1%}")
    print(f"Val Loss: {avg_val_loss:.4f} | Acc: {val_acc:.1%}")
    
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        torch.save(model.state_dict(), 'best_model.pth')
        print("Best model saved!")

print("\n" + "="*70)
print("TRAINING COMPLETE!")
print("="*70)
print(f"Best Val Loss: {best_val_loss:.4f}")
print(f"Final Val Acc: {history['val_acc'][-1]:.1%}")

## 8. Test Evaluation

In [ ]:
# Load best model
model.load_state_dict(torch.load('best_model.pth', map_location=device, weights_only=True))
model.eval()

test_preds, test_labels = [], []
test_emo_preds, test_emo_labels = [], []

with torch.no_grad():
    for batch in tqdm(test_loader, desc='Testing'):
        input_ids = batch['input_ids'].to(device)
        mask = batch['attention_mask'].to(device)
        sent_labels = batch['sentiment'].to(device)
        emo_labels = batch['emotions'].to(device)
        
        with torch.amp.autocast('cuda'):
            sent_logits, emo_logits, _ = model(input_ids, mask)
        
        preds = torch.argmax(sent_logits, 1).cpu()
        test_preds.extend(preds.numpy())
        test_labels.extend(sent_labels.cpu().numpy())
        
        emo_preds = (torch.sigmoid(emo_logits) > 0.5).cpu().numpy()
        test_emo_preds.extend(emo_preds)
        test_emo_labels.extend(emo_labels.cpu().numpy())

from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report

acc = accuracy_score(test_labels, test_preds)
prec, rec, f1, _ = precision_recall_fscore_support(test_labels, test_preds, average='weighted')

print("\n" + "="*70)
print("TEST RESULTS - SENTIMENT")
print("="*70)
print(f"Accuracy: {acc:.4f} ({acc:.1%})")
print(f"Precision: {prec:.4f}")
print(f"Recall: {rec:.4f}")
print(f"F1-Score: {f1:.4f}")
print("\n" + classification_report(test_labels, test_preds,
                                    target_names=['Negative', 'Neutral', 'Positive']))

emo_names = ['Joy', 'Sadness', 'Anger', 'Fear', 'Surprise', 'Neutral']
test_emo_preds = np.array(test_emo_preds)
test_emo_labels = np.array(test_emo_labels)

print("\n" + "="*70)
print("TEST RESULTS - EMOTIONS")
print("="*70)
for i, emo in enumerate(emo_names):
    emo_acc = accuracy_score(test_emo_labels[:, i], test_emo_preds[:, i])
    print(f"{emo:12s}: {emo_acc:.4f} ({emo_acc:.1%})")

## 9. Sample Predictions

In [ ]:
def predict(text):
    model.eval()
    enc = tokenizer(text, max_length=MAX_LEN, padding='max_length',
                   truncation=True, return_tensors='pt')
    input_ids = enc['input_ids'].to(device)
    mask = enc['attention_mask'].to(device)
    
    with torch.no_grad():
        with torch.amp.autocast('cuda'):
            sent_logits, emo_logits, _ = model(input_ids, mask)
    
    sent_probs = torch.softmax(sent_logits.float(), 1)[0]
    sent_names = ['Negative', 'Neutral', 'Positive']
    pred = sent_names[torch.argmax(sent_probs)]
    conf = torch.max(sent_probs).item()
    
    emo_probs = torch.sigmoid(emo_logits.float())[0]
    emo_names = ['Joy', 'Sadness', 'Anger', 'Fear', 'Surprise', 'Neutral']
    top_idx = torch.topk(emo_probs, 3).indices
    top_emos = [(emo_names[i], emo_probs[i].item()) for i in top_idx]
    
    print(f"\nText: {text[:80]}{'...' if len(text) > 80 else ''}")
    print(f"Sentiment: {pred} ({conf:.1%})")
    print(f"Emotions: {', '.join([f'{e}({p:.2f})' for e,p in top_emos])}")
    print("-"*60)

examples = [
    "The protagonist felt overwhelming joy as she reunited with her family.",
    "A deep melancholy settled over him as he gazed upon the ruins.",
    "Fury coursed through her veins when she discovered the betrayal.",
    "The dark corridor filled her with inexplicable dread.",
    "To her astonishment, the letter contained unexpected news.",
    "The story unfolded in a matter-of-fact manner."
]

print("\n" + "="*60)
print("SAMPLE PREDICTIONS")
print("="*60)

for text in examples:
    predict(text)

## 10. Training History

In [ ]:
import matplotlib.pyplot as plt

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(history['train_loss'], 'o-', label='Train')
ax1.plot(history['val_loss'], 's-', label='Val')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.set_title('Loss')
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2.plot(history['train_acc'], 'o-', label='Train')
ax2.plot(history['val_acc'], 's-', label='Val')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Accuracy')
ax2.set_title('Accuracy')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('training_history.png', dpi=150)
plt.show()

print("Saved training_history.png")

## 11. Save Model

In [ ]:
model_cpu = model.cpu()
torch.save({
    'model_state_dict': model_cpu.state_dict(),
    'tokenizer': 'bert-base-uncased',
    'max_len': MAX_LEN,
    'test_accuracy': acc,
    'test_f1': f1,
    'emotion_labels': ['Joy', 'Sadness', 'Anger', 'Fear', 'Surprise', 'Neutral'],
    'sentiment_labels': ['Negative', 'Neutral', 'Positive'],
    'dataset': 'Wagih_ISEAR'
}, 'enhanced_emotion_model.pth')

print("\n" + "="*60)
print("MODEL SAVED!")
print("="*60)
print(f"Test Accuracy: {acc:.1%}")
print(f"Test F1: {f1:.4f}")
print(f"Samples: {len(final_df):,}")
print("\nDownload enhanced_emotion_model.pth for inference")
print("="*60)